In [ ]:
# ===== CELL 1: Install Unsloth =====
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
# ===== CELL 2: Imports & Model Loading =====
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192  # Longer context for big diffs
dtype = None
load_in_4bit = False  # Full precision - you have the VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ Model loaded")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2026.1.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Model loaded


In [ ]:
# ===== CELL 3: Load & Format Dataset =====
import json
from datasets import Dataset

data = []
with open('train_dataset_clean.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} samples")

def format_prompt(sample):
    system_msg = "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."

    user_msg = f"""Review the following code diff and provide feedback:
````diff
{sample['input'][:6000]}
```"""

    assistant_msg = sample['output']

    text = f"""<|im_start|>system
{system_msg}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
{assistant_msg}<|im_end|>"""

    return {"text": text}


#The bug was `<|im_end|}` — should be `<|im_end|>` (closing `>` not `}`).

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['test'])}")




Loaded 12488 samples


Map:   0%|          | 0/12488 [00:00<?, ? examples/s]

Train: 11863, Val: 625


In [ ]:
# ===== CELL 4: Training (Optimized for H100) =====
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,
    packing=True,  # Faster training
    args=TrainingArguments(
        per_device_train_batch_size=4,  # Larger batch - H100 can handle it
        gradient_accumulation_steps=4,  # Effective batch = 16
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,  # H100 loves bf16
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/11863 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/625 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,863 | Num Epochs = 3 | Total steps = 2,226
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: deepanathan-rajendiran (deepanathan-rajendiran-university-at-buffalo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss
100,0.824700,0.823746
200,0.824100,0.791272
300,0.794800,0.764854
400,0.793300,0.746032
500,0.723400,0.726806
600,0.732200,0.710938
700,0.696500,0.694977
800,0.595200,0.687584
900,0.619200,0.678911
1000,0.610000,0.670639


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▇▆▅▄▄▃▃▃▂▂▂▁▁▂▂▂▁▁▁▁▁
eval/runtime,█▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂
eval/samples_per_second,▁██▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇
eval/steps_per_second,▁██▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇███
train/grad_norm,▄▁▁▁▁▁▂▃▁▂▄▂▃▄▂▆▅▅▄▂▄▃▆▄▆▃▄▄▅▄▄█▆▅▃▄▇▄▄▅
train/learning_rate,█████████▇▇▇▇▇▆▆▆▆▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/loss,████▇▇▇▇▆▇▆▆▆▆▅▅▄▄▅▄▄▄▄▄▃▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂
eval/loss,0.6388
eval/runtime,35.0535


In [ ]:
# ===== CELL 5: Test the Model =====
FastLanguageModel.for_inference(model)

test_diff = """@@ -45,7 +45,7 @@ def process_data(self, data):
-        result = data.split(',')
+        result = data.split(',')[0]
         return result"""

messages = [
    {"role": "system", "content": "You are a Senior Software Engineer reviewing code changes."},
    {"role": "user", "content": f"Review this code diff:\n\n```diff\n{test_diff}\n```"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("="*50)
print("MODEL REVIEW:")
print(response.split("assistant")[-1].strip())

# ===== CELL 6: Save =====
model.save_pretrained("code-reviewer-lora")
tokenizer.save_pretrained("code-reviewer-lora")

# Merge for easier deployment
model.save_pretrained_merged("code-reviewer-merged", tokenizer, save_method="merged_16bit")

print("✅ Done!")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


MODEL REVIEW:
Remove the `[0]` index from `result = data.split(',')[0]`. The output of `split(',')` is already a list and indexing it at `[0]` discards the rest of the data. If you want to keep only the first value, explain why that's necessary. Otherwise, just use `result = data.split(',')` as before.
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `code-reviewer-merged`: 100%|██████████| 4/4 [00:35<00:00,  8.78s/it]


Successfully copied all 4 files from cache to `code-reviewer-merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:02<00:00, 15.67s/it]


Unsloth: Merge process complete. Saved to `/content/code-reviewer-merged`
✅ Done!


In [ ]:


"""

---

## Key Changes for H100

| Setting | T4 | H100 |
|---------|-----|------|
| `load_in_4bit` | True | **False** |
| `r` (LoRA rank) | 16 | **32** |
| `batch_size` | 2 | **8** |
| `max_seq_length` | 4096 | **8192** |
| `packing` | False | **True** |
| `bf16` | Maybe | **True** |

---

## Expected Time on H100

~20-30 minutes for 12k samples, 3 epochs.

---

Run it and share the training loss curve when it finishes.

SyntaxError: incomplete input (ipython-input-3467765799.py, line 1)

In [ ]:
# ===== Test the Model =====
FastLanguageModel.for_inference(model)

test_diffs = [
    # Test 1: Security issue
    """@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)""",

    # Test 2: Bug
    """@@ -5,7 +5,7 @@ def get_user(user_id):
-        user = db.query(User).filter(User.id == user_id).first()
+        user = db.query(User).filter(User.id == user_id)
         return user.name""",

    # Test 3: Style
    """@@ -1,5 +1,5 @@
-def calculateTotalPrice(items):
+def calc(i):
     total = 0
-    for item in items:
+    for x in i:
         total += x.price""",
]

for i, diff in enumerate(test_diffs, 1):
    messages = [
        {"role": "system", "content": "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."},
        {"role": "user", "content": f"Review this code diff:\n\n```diff\n{diff}\n```"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    review = response.split("assistant")[-1].strip()

    print(f"\n{'='*50}")
    print(f"TEST {i}")
    print(f"{'='*50}")
    print(f"DIFF:\n{diff}")
    print(f"\nMODEL REVIEW:\n{review}")

In [ ]:
FastLanguageModel.for_inference(model)

test_diffs = [
    """@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)""",
]

for diff in test_diffs:
    messages = [
        {"role": "system", "content": "You are a Senior Software Engineer. Review code for bugs, security issues, and bad practices. Explain WHY something is problematic and suggest the correct fix."},
        {"role": "user", "content": f"This code change was submitted for review. Identify any problems and explain why they are issues:\n\n```diff\n{diff}\n```"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=300,
        temperature=0.3,
        top_p=0.9,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response.split("assistant")[-1].strip())

In [ ]:
import json
from tqdm import tqdm

# Load your clean dataset
data = []
with open('train_dataset_clean.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} samples")

# Put model in inference mode
FastLanguageModel.for_inference(model)

# Generate rejected responses
dpo_data = []

for sample in tqdm(data[:5000], desc="Generating DPO pairs"):  # Start with 5k
    diff = sample['input'][:4000]
    human_review = sample['output']

    # Generate model's response (this will be "rejected")
    messages = [
        {"role": "system", "content": "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."},
        {"role": "user", "content": f"Review this code diff:\n\n```diff\n{diff}\n```"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    model_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    model_review = model_response.split("assistant")[-1].strip()

    # Skip if model response is too similar to human (no learning signal)
    if model_review.lower()[:50] == human_review.lower()[:50]:
        continue

    # Skip if model response is empty
    if len(model_review) < 20:
        continue

    dpo_data.append({
        "prompt": diff,
        "chosen": human_review,
        "rejected": model_review,
    })

print(f"\n✅ Generated {len(dpo_data)} DPO pairs")

# Save
with open('dpo_dataset.jsonl', 'w') as f:
    for item in dpo_data:
        f.write(json.dumps(item) + '\n')

Loaded 12488 samples


Generating DPO pairs: 100%|██████████| 5000/5000 [2:48:17<00:00,  2.02s/it]


✅ Generated 4961 DPO pairs


In [ ]:
# ===== CELL 2: Load DPO Dataset =====
import json
from datasets import Dataset

dpo_data = []
with open('dpo_dataset.jsonl', 'r') as f:
    for line in f:
        dpo_data.append(json.loads(line))

print(f"✅ Loaded {len(dpo_data)} DPO pairs")


# Format for DPO
def format_dpo(sample):
    system_msg = "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."
    user_msg = f"Review this code diff:\n\n```diff\n{sample['prompt'][:4000]}\n```"

    prompt = f"""<|im_start|>system
{system_msg}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
"""
    return {
        "prompt": prompt,
        "chosen": sample["chosen"] + "<|im_end|>",
        "rejected": sample["rejected"] + "<|im_end|>",
    }

dataset = Dataset.from_list(dpo_data)
dataset = dataset.map(format_dpo)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train: {len(dataset['train'])}, Val: {len(dataset['test'])}")

✅ Loaded 4961 DPO pairs


Map:   0%|          | 0/4961 [00:00<?, ? examples/s]

Train: 4712, Val: 249


In [ ]:
#===== CELL 3: Load Model Fresh for DPO =====
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

# Load BASE model first
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=False,
)

# Load your SFT adapter on top
model = PeftModel.from_pretrained(
    model,
    "code-reviewer-lora",
    is_trainable=True,  # Important: make it trainable for DPO
)

print("✅ Model loaded with SFT adapter")


==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: e0d9d94d-9be7-4659-a336-733b863f98ac)')' thrown while requesting HEAD https://huggingface.co/unslothai/colabpro/resolve/234f33d5f3e1d9ad83421f33640cd88474a25025/config.json
Retrying in 1s [Retry 1/5].


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded with SFT adapter


In [ ]:
# ===== CELL 4: DPO Training =====
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-5,
        bf16=True,
        logging_steps=25,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs-dpo",
        eval_strategy="steps",
        eval_steps=100,
        remove_unused_columns=False,
    ),
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

print("🚀 Starting DPO training...")
dpo_trainer.train()


Extracting prompt in train dataset (num_proc=16):   0%|          | 0/4712 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=16):   0%|          | 0/4712 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=16):   0%|          | 0/4712 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=16):   0%|          | 0/249 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=16):   0%|          | 0/249 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,712 | Num Epochs = 1 | Total steps = 589
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
100,0.331400,0.368304,2.247904,0.440871,0.837302,1.807033,-80.284012,-80.695877,-0.466996,-0.447112,0,0,0
200,0.264500,0.303083,1.924293,-0.605001,0.873016,2.529294,-83.520126,-91.154610,-0.171421,-0.146048,No Log,No Log,No Log
300,0.232900,0.267381,1.952594,-0.857589,0.904762,2.810183,-83.237122,-93.680489,-0.082960,-0.059670,No Log,No Log,No Log
400,0.221700,0.241066,1.504886,-1.522303,0.904762,3.027189,-87.714188,-100.327621,-0.027624,0.001490,No Log,No Log,No Log
500,0.218600,0.233791,0.009763,-3.388346,0.912698,3.398109,-102.665421,-118.988068,-0.097814,-0.086383,No Log,No Log,No Log


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 77f41808-01b9-4f75-99dd-11923891738f)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-7B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].


TrainOutput(global_step=589, training_loss=0.33317447761121144, metrics={'train_runtime': 1888.4334, 'train_samples_per_second': 2.495, 'train_steps_per_second': 0.312, 'total_flos': 0.0, 'train_loss': 0.33317447761121144, 'epoch': 1.0})

In [ ]:
# ===== CELL 5: Save DPO Model =====
model.save_pretrained("code-reviewer-dpo")
tokenizer.save_pretrained("code-reviewer-dpo")
print("✅ DPO model saved!")

# ===== CELL 6: Test DPO Model =====
model.eval()

test_diffs = [
    # Security issue
    """@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)""",

    # Bug
    """@@ -5,7 +5,7 @@ def get_user(user_id):
-        user = db.query(User).filter(User.id == user_id).first()
+        user = db.query(User).filter(User.id == user_id)
         return user.name""",

    # Style
    """@@ -1,5 +1,5 @@
-def calculateTotalPrice(items):
+def calc(i):
     total = 0
-    for item in items:
+    for x in i:
         total += x.price""",
]

for i, diff in enumerate(test_diffs, 1):
    messages = [
        {"role": "system", "content": "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."},
        {"role": "user", "content": f"Review this code diff:\n\n```diff\n{diff}\n```"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    review = response.split("assistant")[-1].strip()

    print(f"\n{'='*50}")
    print(f"TEST {i}")
    print(f"{'='*50}")
    print(f"DIFF:\n{diff}")
    print(f"\nMODEL REVIEW:\n{review}")


✅ DPO model saved!

TEST 1
DIFF:
@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)

MODEL REVIEW:
1.  **Sridhar, please provide a justification for the `0.99` value in the `dbscan` algorithm.** 2.  **If the `0.99` value is a hyperparameter, document the expected range and optimization strategy.** 3.  **If the `0.99` value is a domain-specific threshold, explain the derivation of the value, including the underlying business problem, data distribution, and the rationale for this specific number.** 4.  **For all other values, if they are not hyperparameters, add a comment to the code explaining the source or purpose of the value.** 5.  **Sridhar, please check the code and update the comments as per the above instructions.** 6.  **Sridhar, please also check the code and update the comments as per the above instructions.** 7.  **Sridhar, please check the code and update the comments as per t

In [ ]:

# ===== CELL 7: Download Model =====
from google.colab import files
!zip -r code-reviewer-dpo.zip code-reviewer-dpo/
files.download('code-reviewer-dpo.zip')

  adding: code-reviewer-dpo/ (stored 0%)
  adding: code-reviewer-dpo/special_tokens_map.json (deflated 67%)
  adding: code-reviewer-dpo/merges.txt (deflated 57%)
  adding: code-reviewer-dpo/added_tokens.json (deflated 65%)
  adding: code-reviewer-dpo/tokenizer_config.json (deflated 89%)
  adding: code-reviewer-dpo/adapter_config.json (deflated 57%)
  adding: code-reviewer-dpo/README.md (deflated 65%)
  adding: code-reviewer-dpo/adapter_model.safetensors (deflated 7%)
  adding: code-reviewer-dpo/vocab.json (deflated 61%)
  adding: code-reviewer-dpo/chat_template.jinja (deflated 71%)
  adding: code-reviewer-dpo/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Reload your SFT model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="code-reviewer-lora",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=False,
)

FastLanguageModel.for_inference(model)

# Better prompt that forces critical thinking
test_diff = """@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)"""

messages = [
    {"role": "system", "content": """You are a Senior Security Engineer reviewing code.
Your job is to REJECT dangerous changes. Look for:
- Hardcoded secrets/passwords (CRITICAL)
- SQL injection risks
- Missing input validation
Be direct: if something is wrong, say REJECT and explain why."""},
    {"role": "user", "content": f"Review this change. Is it safe to merge?\n\n```diff\n{test_diff}\n```"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=200,
    temperature=0.2,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1])

==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 8d4716d2-c972-454e-b63b-5b6135f9695d)')' thrown while requesting HEAD https://huggingface.co/unslothai/colabpro/resolve/234f33d5f3e1d9ad83421f33640cd88474a25025/.gitattributes
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 8c1fc066-f2f0-4d61-a22d-b1132b60994b)')' thrown while requesting HEAD https://huggingface.co/unslothai/repeat/resolve/7c48478c02f84ed89f149b0815cc0216ee831fb0/.gitattributes
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 239e9385-c611-4af6-bf86-bf7d5a98796b)')' thrown while requesting HEAD https://huggingface.co/unslothai/repeat/resolve/7c48478c02f84ed89f149b0815cc0216ee831fb0/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 007968ef-e6c7-4509-9d39-72c015f3e64d)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-7B-Instruct/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 383f2d4e-6ade-411f-89a9-56593a55a75b)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-7B-Instruct/resolve/main/generation_config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 871f89e2-d7c4-434e-b906-0bbe173df675)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-7B-Instruct/resolve/main/generation_config.json
Retrying in 4s [Retry 3/5].



REJECT: This PR introduces a hardcoded password (`password = "admin123"`) which is a security risk. Ensure passwords are never hard-coded in source code.


In [ ]:
# ===== CELL 1: Load DPO Dataset =====
import json
from datasets import Dataset

dpo_data = []
with open('dpo_dataset.jsonl', 'r') as f:
    for line in f:
        dpo_data.append(json.loads(line))

print(f"✅ Loaded {len(dpo_data)} DPO pairs")

def format_dpo(sample):
    system_msg = "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."
    user_msg = f"Review this code diff:\n\n```diff\n{sample['prompt'][:4000]}\n```"

    prompt = f"""<|im_start|>system
{system_msg}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
"""
    return {
        "prompt": prompt,
        "chosen": sample["chosen"] + "<|im_end|>",
        "rejected": sample["rejected"] + "<|im_end|>",
    }

dataset = Dataset.from_list(dpo_data)
dataset = dataset.map(format_dpo)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train: {len(dataset['train'])}, Val: {len(dataset['test'])}")

# ===== CELL 2: Load Base Model + SFT Adapter =====
from unsloth import FastLanguageModel
from peft import PeftModel
import torch
import copy

# Load base model
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=False,
)

# Load SFT adapter (trainable)
model = PeftModel.from_pretrained(
    base_model,
    "code-reviewer-lora",
    is_trainable=True,
)

print("✅ Model loaded")

# ===== CELL 3: DPO Training (Conservative) =====
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Uses implicit reference
    args=DPOConfig(
        # Batch settings
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # CONSERVATIVE settings to prevent collapse
        num_train_epochs=1,
        # Just change these in the DPOConfig:
        learning_rate=1e-6,
        max_steps=500,
        beta=0.05,

        # Standard settings
        warmup_ratio=0.1,
        bf16=True,
        logging_steps=10,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs-dpo-v2",

        # Eval
        eval_strategy="steps",
        eval_steps=50,

        # Important
        remove_unused_columns=False,
        max_length=2048,
        max_prompt_length=1500,
    ),
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

print("🚀 Starting DPO training (conservative settings)...")
dpo_trainer.train()

# ===== CELL 4: Save =====
model.save_pretrained("code-reviewer-dpo-v2")
tokenizer.save_pretrained("code-reviewer-dpo-v2")
print("✅ DPO model saved!")

# ===== CELL 5: Test =====
model.eval()

test_diffs = [
    # Security issue
    """@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)""",

    # Bug
    """@@ -5,7 +5,7 @@ def get_user(user_id):
-        user = db.query(User).filter(User.id == user_id).first()
+        user = db.query(User).filter(User.id == user_id)
         return user.name""",

    # Style
    """@@ -1,5 +1,5 @@
-def calculateTotalPrice(items):
+def calc(i):
     total = 0
-    for item in items:
+    for x in i:
         total += x.price""",
]

for i, diff in enumerate(test_diffs, 1):
    messages = [
        {"role": "system", "content": "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."},
        {"role": "user", "content": f"Review this code diff:\n\n```diff\n{diff}\n```"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    review = response.split("assistant")[-1].strip()

    print(f"\n{'='*50}")
    print(f"TEST {i}")
    print(f"{'='*50}")
    print(f"DIFF:\n{diff}")
    print(f"\nMODEL REVIEW:\n{review}")

✅ Loaded 4961 DPO pairs


Map:   0%|          | 0/4961 [00:00<?, ? examples/s]

Train: 4712, Val: 249
==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ebdf68e0-6d94-4bc4-b62e-67071b37a66e)')' thrown while requesting HEAD https://huggingface.co/unslothai/colabpro/resolve/234f33d5f3e1d9ad83421f33640cd88474a25025/config.json
Retrying in 1s [Retry 1/5].


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

🚀 Starting DPO training (conservative settings)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,712 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
50,0.812300,0.770500,1.722838,1.602639,0.488095,0.120199,-68.033875,-52.730251,-0.080717,-0.072856,0,0,0
100,0.725800,0.757719,1.728129,1.585639,0.507937,0.142490,-67.928062,-53.070251,-0.078004,-0.066422,No Log,No Log,No Log
150,0.807000,0.747596,1.731736,1.572154,0.507937,0.159582,-67.855927,-53.339954,-0.072692,-0.057385,No Log,No Log,No Log
200,0.796400,0.739528,1.730297,1.557377,0.503968,0.172920,-67.884697,-53.635498,-0.067379,-0.050098,No Log,No Log,No Log
250,0.746100,0.730900,1.732971,1.545248,0.523810,0.187723,-67.831207,-53.878071,-0.064737,-0.045664,No Log,No Log,No Log
300,0.694900,0.725410,1.732045,1.534713,0.535714,0.197332,-67.849739,-54.088772,-0.066164,-0.045472,No Log,No Log,No Log
350,0.678600,0.721545,1.730871,1.526943,0.539683,0.203928,-67.873215,-54.244175,-0.064944,-0.043076,No Log,No Log,No Log
400,0.730900,0.717411,1.730368,1.519120,0.535714,0.211248,-67.883286,-54.400627,-0.064821,-0.042314,No Log,No Log,No Log
450,0.759500,0.716432,1.730462,1.517258,0.543651,0.213204,-67.881393,-54.437878,-0.065949,-0.043249,No Log,No Log,No Log
500,0.854100,0.715405,1.730182,1.516014,0.539683,0.214168,-67.887009,-54.462749,-0.065657,-0.042730,No Log,No Log,No Log


✅ DPO model saved!

TEST 1
DIFF:
@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)

MODEL REVIEW:
Remove the `os.environ.get('DB_PASS')` line and set `password` to `"admin123"`.

TEST 2
DIFF:
@@ -5,7 +5,7 @@ def get_user(user_id):
-        user = db.query(User).filter(User.id == user_id).first()
+        user = db.query(User).filter(User.id == user_id)
         return user.name

MODEL REVIEW:
Ensure the `get_user` function returns a string representing the user's name, not the entire User object.

TEST 3
DIFF:
@@ -1,5 +1,5 @@
-def calculateTotalPrice(items):
+def calc(i):
     total = 0
-    for item in items:
+    for x in i:
         total += x.price

MODEL REVIEW:
Rename the variable `item` to `x` and rename the function parameter from `items` to `i`.


In [ ]:
# Save adapter
model.save_pretrained("code-reviewer-lora-final")
tokenizer.save_pretrained("code-reviewer-lora-final")

# Zip and download
!zip -r code-reviewer-lora-final.zip code-reviewer-lora-final/
from google.colab import files
files.download('code-reviewer-lora-final.zip')

print("✅ LoRA adapter saved!")

  adding: code-reviewer-lora-final/ (stored 0%)
  adding: code-reviewer-lora-final/special_tokens_map.json (deflated 67%)
  adding: code-reviewer-lora-final/merges.txt (deflated 57%)
  adding: code-reviewer-lora-final/added_tokens.json (deflated 65%)
  adding: code-reviewer-lora-final/tokenizer_config.json (deflated 89%)
  adding: code-reviewer-lora-final/adapter_config.json (deflated 57%)
  adding: code-reviewer-lora-final/README.md (deflated 65%)
  adding: code-reviewer-lora-final/adapter_model.safetensors (deflated 7%)
  adding: code-reviewer-lora-final/vocab.json (deflated 61%)
  adding: code-reviewer-lora-final/chat_template.jinja (deflated 71%)
  adding: code-reviewer-lora-final/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ LoRA adapter saved!


In [ ]:
from unsloth import FastLanguageModel

# Load your SFT model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="code-reviewer-lora",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=False,
)

# Merge LoRA into base model and save
model.save_pretrained_merged(
    "code-reviewer-merged",
    tokenizer,
    save_method="merged_16bit",  # or "merged_4bit" for smaller size
)

print("✅ Merged model saved!")

# Zip (will be large)
!zip -r code-reviewer-merged.zip code-reviewer-merged/

==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `code-reviewer-merged`: 100%|██████████| 4/4 [00:33<00:00,  8.42s/it]


Successfully copied all 4 files from cache to `code-reviewer-merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:01<00:00, 15.40s/it]


Unsloth: Merge process complete. Saved to `/content/code-reviewer-merged`
✅ Merged model saved!
  adding: code-reviewer-merged/ (stored 0%)
  adding: code-reviewer-merged/special_tokens_map.json (deflated 67%)
  adding: code-reviewer-merged/merges.txt (deflated 57%)
  adding: code-reviewer-merged/added_tokens.json (deflated 65%)
  adding: code-reviewer-merged/tokenizer_config.json (deflated 83%)
  adding: code-reviewer-merged/model.safetensors.index.json (deflated 95%)
  adding: code-reviewer-merged/config.json (deflated 71%)
  adding: code-reviewer-merged/model-00003-of-00004.safetensors (deflated 21%)
  adding: code-reviewer-merged/model-00004-of-00004.safetensors (deflated 21%)
  adding: code-reviewer-merged/.cache/ (stored 0%)
  adding: code-reviewer-merged/.cache/huggingface/ (stored 0%)
  adding: code-reviewer-merged/.cache/huggingface/.gitignore (stored 0%)
  adding: code-reviewer-merged/.cache/huggingface/download/ (stored 0%)
  adding: code-reviewer-merged/.cache/huggingface/d

In [ ]:
from unsloth import FastLanguageModel

# Load your SFT model
model, tokenizer = FastLanguageModel.from_pretrained()